In [ ]:
%matplotlib inline

In [ ]:
# Import your libraries here

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import norm
import os
import sys

In [ ]:
current_dir = os.path.abspath('')
project_root = os.path.abspath(os.path.join(current_dir, '../../'))

if project_root not in sys.path:
    sys.path.append(project_root)

os.chdir(project_root)

print(f"Working directory set to: {os.getcwd()}")

In [ ]:
# Import your modules here

from src.mathematical_calculations_utils import option_calc_utils as o_calc
from src.mathematical_calculations_utils import bond_calc_utils as b_calc
from src.plotting_utils import plotting_utils as plot_utils

# Transition: From Portfolio Optimization to Derivative Pricing

Up to this point, the project focused on optimizing portfolios of risky assets. However, financial markets also allow investors to reshape portfolio risk using derivative instruments.

Options provide nonlinear payoffs that can protect portfolios against downside risk or introduce asymmetric exposures.

The next section, therefore, moves from portfolio optimization to option pricing. We will look at the most basic version of the Black-Scholes Option Pricing Model - the one that evaluates **European-style** options, which can be exercised only on the expiry date. The mathematics behind the so-called **American-style** options becomes very complicated very fast. 

# Black–Scholes Model and Portfolio Protection

This notebook introduces option pricing using the Black–Scholes model and applies it to portfolio protection strategies.

The analysis also examines option sensitivities (Greeks), which represent the first and second derivatives of the option price with respect to key variables, as well as the first-order sensitivities to the remaining inputs in the model - portfolio variance, time/term of the option, and risk-free rate.

## The Mathematical Foundations of Black–Scholes Option Pricing

The Black–Scholes–Merton model revolutionized finance by showing that an option can be **replicated and perfectly hedged** using a portfolio of the underlying asset and cash. Instead of forecasting the future direction of the stock price, the model derives the option price from **arbitrage-free dynamics** as described [here](https://www.investopedia.com/terms/b/blackscholes.asp).

The key insight is that by continuously adjusting the position in the underlying asset (delta hedging), a portfolio consisting of the option and the underlying can be made **locally risk-free**, and therefore must earn the **risk-free interest rate $r$**.

---

### 1. The Black–Scholes Partial Differential Equation

The no-arbitrage condition leads to the famous Black–Scholes partial differential equation:

$$
\frac{\partial V}{\partial t} + \frac{1}{2}\sigma^2 S^2 \frac{\partial^2 V}{\partial S^2} + rS \frac{\partial V}{\partial S} - rV = 0
$$

Where:

- $\frac{\partial V}{\partial t}$ (Theta) represents the **time decay** of the option.
- $\frac{\partial V}{\partial S}$ relates to **Delta**, the sensitivity of the option price to the underlying.
- $\frac{\partial^2 V}{\partial S^2}$ (Gamma) captures the **curvature** or convexity of the option price.
- $\sigma$ is the **volatility**, the standard deviation of the asset's log returns.

This equation describes how the option value evolves through **time, volatility, and the underlying asset price**.

---

### 2. The Closed-Form Solution for a European Call Option

Solving the PDE with the terminal payoff condition

$$
\max(S-K,0)
$$

yields the Black–Scholes formula for a European call option:

$$
C(S,t) = S N(d_1) - K e^{-rT} N(d_2)
$$

where

$$
d_1 = \frac{\ln(S/K) + (r + \sigma^2/2)T}{\sigma\sqrt{T}}
$$

$$
d_2 = d_1 - \sigma\sqrt{T}
$$

and $N(\cdot)$ is the cumulative distribution function of the **standard normal distribution**.

Interpretation:

- $N(d_1)$ corresponds to the **Delta** of the option — the sensitivity of the option value to the underlying price.
- $N(d_2)$ represents the **risk-neutral probability** that the option expires **in-the-money**.

## Assumptions of the Black–Scholes Model

The Black–Scholes option pricing framework relies on several idealized assumptions:

- The underlying asset follows a geometric Brownian motion.
- Volatility is constant over the life of the option.
- Markets are frictionless and allow continuous trading.
- Interest rates remain constant.

Real markets frequently violate these assumptions, leading to phenomena such as the volatility smile and time-varying implied volatility.

## Hedging the portfolio of stocks. Pricing the option, we are going to use for the hedge

**1.** Let's start by defining some portfolio parameters

In [ ]:
# 1. Define Portfolio Parameters
current_value = 1000000  # $1M Portfolio
strike = current_value * 0.95  # 5% Downside protection (95% floor)
time_to_maturity = 1  # 1 year
rf_rate = 0.03  # 3% risk-free rate

optimal_portfolio_metrics_df = pd.read_csv("data/optimal_portfolio_metrics.csv")
optimal_portfolio_return = optimal_portfolio_metrics_df["Expected Return"].iloc[0]
optimal_portfolio_volatility = optimal_portfolio_metrics_df["Volatility"].iloc[0]

## Black-Scholes Option Pricing Model:

**2.** Let's apply the standard Black-Scholes Option Pricing Model for a European-style put option on the optimal portfolio from notebook 1_3

In [ ]:
# 2. Calculate the price of the hedge - a portfolio put option as defined in point 1
insurance_cost = o_calc.black_scholes(
    S=current_value, 
    K=strike, 
    T=time_to_maturity, 
    r=rf_rate, 
    sigma=optimal_portfolio_volatility, 
    option_type='put'
)

print(f"The value of the portfolio hedge as defined is {insurance_cost:.2f} USD")

## Option Sensitivities (Greeks)

Option pricing models allow us to measure how option values change with respect to underlying variables.

Delta represents the first derivative of the option price with respect to the underlying asset price:

$$
\Delta = \frac{\partial C}{\partial S}
$$

Gamma represents the second derivative:

$$
\Gamma = \frac{\partial^2 C}{\partial S^2}
$$

These sensitivities highlight the nonlinear structure of option payoffs and connect directly to the concept of curvature examined earlier through portfolio variance and bond convexity.

**3.** Let's first plot the option price as a function of the portfolio value in a selected range, e.g., between USD 500,000 and USD 1,000,000.

In [ ]:
current_value_range = np.linspace(500000, 1500000, 100000)

insurance_cost_range =o_calc.black_scholes(
    S=current_value_range, 
    K=strike, 
    T=time_to_maturity, 
    r=rf_rate, 
    sigma=optimal_portfolio_volatility, 
    option_type='put'
)

payoff = np.maximum(strike - current_value_range, 0)

print(insurance_cost_range)

In [ ]:
fig = plot_utils.plot_taylor_expansion(current_value_range,
                                       insurance_cost_range,
                                       current_value,
                                       insurance_cost,
                                       'Portfolio Put Option Price Function',
                                       'Current Portfolio Value',
                                       'Put Option Value/Cost to Insure the Portfolio at $950,000',
                                       payoff_values=payoff,
                                       vline_x=strike,)

plt.show()

**4.** Let's calculate the first and second derivatives numerically using the Black-Scholes formulation outlined above. 

In [ ]:
current_value_target = 1000000

put_price_target, slope, curvature = o_calc.calculate_option_derivatives_1(current_value_target,
                                                                           K=strike,
                                                                           T=time_to_maturity,
                                                                           r=rf_rate,
                                                                           sigma=optimal_portfolio_volatility,
                                                                           option_type='put')
print(f"Put price: {put_price_target:.2f}")
print(f"Delta (slope): {slope:.4f}")
print(f"Gamma (curvature): {curvature}")

**5.** Let us now plot the option price function and its derivatives

In [ ]:
tangent_line = b_calc.draw_tangent_line(current_value_range,
                                        current_value_target,
                                        put_price_target,
                                        slope)

quadratic_approx = b_calc.draw_convex_line(current_value_range,
                                           current_value_target,
                                           put_price_target, slope,
                                           curvature)

fig = plot_utils.plot_taylor_expansion(current_value_range,
                                       insurance_cost_range,
                                       current_value,
                                       insurance_cost,
                                       'Portfolio Put Option Price Function',
                                       'Current Portfolio Value',
                                       'Put Option Value/Cost to Insure the Portfolio at $950,000',
                                       tangent_values=tangent_line,
                                       quadratic_values=quadratic_approx,
                                       payoff_values=payoff,
                                       vline_x=strike,)

plt.show()

## Secondary Option Greeks ##

Up to this point, we focused on the **price function of the put option with respect to the underlying portfolio value** $S$.

At the analysis point $S_0 = 1{,}000{,}000$ USD, we computed:

* the **first derivative** of the price function — **Delta**
* the **second derivative** — **Gamma**

We then visualized these results by drawing:

* the **tangent line** to the option price curve (the linear approximation driven by Delta), and
* the **quadratic curvature** around the point (the second-order approximation driven by Gamma).

But…

**wait a moment. Stop and observe.**

The option pricing function we are working with is not a function of a single variable. In the Black–Scholes framework, the option price is:

$$
P = P(S, \sigma, r, T)
$$

where

* $S$ — underlying asset (portfolio) value
* $\sigma$ — volatility of the underlying
* $r$ — risk-free interest rate
* $T$ — time to maturity

If the option price depends on these variables, then the same logic from calculus applies:

> we can take **partial derivatives** with respect to each of them.

This leads to additional **Option Greeks**, which measure the sensitivity of the option price to changes in these parameters.


### Vega — Sensitivity to Volatility

Vega measures how the option price changes when the **volatility of the underlying asset** changes.

$$
\text{Vega} = \frac{\partial P}{\partial \sigma}
$$

In the Black–Scholes model:

$$
\text{Vega} = S , \phi(d_1)\sqrt{T}
$$

where

$$
d_1 =
\frac{\ln(S/K) + (r + \tfrac12\sigma^2)T}{\sigma\sqrt{T}}
$$

and

$$
\phi(d_1) = \frac{1}{\sqrt{2\pi}}e^{-d_1^2/2}
$$

is the **probability density function of the standard normal distribution**.

**Interpretation:**
Higher volatility increases the probability of extreme outcomes, which increases the value of optionality.


### Rho — Sensitivity to Risk-free Interest Rates ###

Rho measures how the option price changes when the **risk-free interest rate** changes.

$$
\text{Rho} = \frac{\partial P}{\partial r}
$$

For a European **put option** in Black–Scholes:

$$
\text{Rho}_{put} = -K T e^{-rT} N(-d_2)
$$

where

$$
d_2 = d_1 - \sigma\sqrt{T}
$$

and (N(x)) is the **cumulative distribution function** of the standard normal distribution.

**Interpretation:**
For puts, higher interest rates generally **reduce the present value of the strike**, decreasing the option value. For calls, the relationship works in the opposite way.


### Theta — Sensitivity to Time ###

Theta measures how the option price changes as **time passes**, holding everything else constant. This is also known as time decay of the option. The less time remains till the option expiry, the less influence of the time variable over the price of the option.

$$
\Theta = \frac{\partial P}{\partial T}
$$

For a European **put option**:

$$
\Theta = -\frac{S \phi(d_1)\sigma}{2\sqrt{T}} * r K e^{-rT} N(-d_2)
$$

**Interpretation:**
Theta captures **time decay**. As maturity approaches, the value of optionality tends to decline because there is less time for favorable price movements to occur.

### A Multidimensional Sensitivity View

We started with the first and second derivatives of the price function with respect to the price of the underlying:

$$
P = P(S)
$$

to a **multivariable function**

$$
P = P(S, \sigma, r, T)
$$

with a full set of sensitivities:

| Greek | Derivative                           | Economic Meaning                |
| ----- | ------------------------------------ | ------------------------------- |
| Delta | $\frac{\partial P}{\partial S}$      | Sensitivity to underlying price |
| Gamma | $\frac{\partial^2 P}{\partial S^2}$  | Curvature of the price function |
| Vega  | $\frac{\partial P}{\partial \sigma}$ | Sensitivity to volatility       |
| Rho   | $\frac{\partial P}{\partial r}$      | Sensitivity to interest rates   |
| Theta | $\frac{\partial P}{\partial T}$      | Sensitivity to time             |

In other words, the option price surface is **not a simple curve**, but a **multidimensional object**, and the Greeks describe the **local geometry of that surface**.


### Additional Note ###

Delta, Gamma, as well as Vega, Rho, and Theta, are by far not an exhaustive list of functional analysis derivatives we can calculate. For academic purposes, I have come across a larger list that also includes "Greeks" as follows:

 - **Volga/Vomma** measures curvature w.r.t. volatility (how Vega changes when vol changes).

 - **Vanna** is a cross-derivative (how Vega changes with price, or how Delta changes with vol).

 - **Charm** is how Delta changes with time.

The list also includes **Speed**  $ \frac{\partial \Gamma}{\partial S} $, **Zomma** $ \frac{\partial \Gamma}{\partial \sigma} $, and **Color** $ \frac{\partial \Gamma}{\partial T} $, which represent higher-order sensitivities describing how the curvature of the option price function itself changes with respect to the underlying price, volatility, and time to maturity.

**6.** Let us calculate $\rho$, $\nu$, and $\theta$ of our optimal portfolio put option hedge

In [ ]:
vega, rho, theta = o_calc.calculate_secondary_option_greeks(
    current_value,
    strike,
    time_to_maturity,
    rf_rate,
    optimal_portfolio_volatility,
    option_type='put'
)

print(f"Vega: {vega:.4f}\nRho: {rho:.4f}\nTheta: {theta:.4f}")

### Interpretation of the Secondary Greeks

At the analysis point $S_0 = \$1{,}000{,}000$, the calculated sensitivities describe how the option price reacts to small changes in key model parameters.

When I ran the algorithm I received the following results:

**1. Vega = 356,279**
$$
\text{Vega} = \frac{\partial P}{\partial \sigma}$$

Vega measures sensitivity to volatility and a **1% increase in volatility** $Delta\sigma = 0.01$ increases the option price by approximately:

ΔP≈356,279×0.01≈$3,563\Delta P \approx 356{,}279 \times 0.01 \approx \$3{,}563

This reflects the fact that higher uncertainty increases the value of downside protection.

**2. Rho = −369,168**

Rho=∂P∂r \text{Rho} = \frac{\partial P}{\partial r} 

Rho measures sensitivity to the risk-free interest rate and a **1% increase in interest rates** decreases the option value by approximately:

ΔP≈−369,168×0.01≈−$3,692\Delta P \approx -369{,}168 \times 0.01 \approx -\$3{,}692

For put options, higher interest rates reduce the present value of the strike payoff and for call options it works in the other direction

**3. Theta = −28,718**

Θ=∂P∂T\Theta = \frac{\partial P}{\partial T}

Theta measures sensitivity to time. As time passes, the option gradually loses value and we can calculate the daily time decay of our optimal portfolio hedge as follows:

ΔT=−1365\Delta T = -\frac{1}{365}

ΔP≈−28,718×(−1365)≈−$79\Delta P \approx -28{,}718 \times \left(-\frac{1}{365}\right) \approx -\$79

Thus the option loses roughly **\$79 per day** due to time decay.

These sensitivities illustrate that the option price is a **multivariable function**

P=P(S,σ,r,T) P = P(S,\sigma,r,T) 

Vega, Rho, and Theta are key in order to be able to describe how the option value reacts to changes in **volatility, interest rates, and time** or to explain the overall price behavior of the option contract. As a result, we can conclude that the option price function is multidimensional and a powerful example of how linear algebra and geometry find real-world applications with multi-trillion-dollar implications.

### Protective Put Structure

We combine our long portfolio position ($S$) with a **long put option** ($P$).  
This strategy creates a **floor for the portfolio value**, limiting downside risk while preserving upside potential. In payoff terms, the combined position behaves like a **synthetic call option** on the portfolio.

### Mathematical Payoff

The **gross payoff at expiry ($T$)** is:

$$
V_T = S_T + \max(K - S_T, 0)
$$

Where:

- $S_T$ — Portfolio value at expiry  
- $K$ — Strike price, representing the **minimum floor value** we are willing to accept  
- $P_0$ — Upfront premium (insurance cost) paid for the put option, calculated using the **Black–Scholes model**

In [ ]:
# 1) Portfolio value
stock_payoff = current_value_range

# 2) Put payoff at expiry
put_payoff = np.maximum(strike - current_value_range, 0)

# 3) Protective put payoff (gross)
protective_put_payoff = stock_payoff + put_payoff

# 4) Protective put payoff net of premium paid today
protective_put_net = protective_put_payoff - insurance_cost

# 5) Plot the Protective put payoff net of premium paid today
plt.figure(figsize=(10,5))

# add plot lines
plt.plot(current_value_range, stock_payoff, label='Unprotected Portfolio: $V(S_T)=S_T$', color='navy', lw=2)
plt.plot(current_value_range, protective_put_payoff, linestyle='--', label='Protective Put (Gross): $S_T + (K-S_T)^+$', color='orange', lw=2)
plt.plot(current_value_range, protective_put_net, linestyle=':', label=f'Protective Put (Net of Premium): $S_T + (K-S_T)^+ - P_0$ (P₀={insurance_cost:,.0f})', color='dodgerblue', lw=2.5)

plt.axvline(strike, color='red', linestyle='--', label=f'Strike $K={strike:,.0f}$')

# mark the analysis point at expiry reference (optional visual anchor)
plt.scatter([current_value], [current_value], color='red', label=f'Current Value $S_0={current_value:,.0f}$')

# settings for the output graph
plt.xlabel('Portfolio Value at Expiry $S_T$')
plt.ylabel('Payoff / Portfolio Value')
plt.title('Protective Put: Portfolio Payoff Shape and the Effect of Premium')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Conclusion: From Asset Allocation to Risk Architecture

In this notebook we moved through the full quantitative pipeline that connects portfolio construction with derivatives-based risk management.

**Optimization.**  
Using the Markowitz framework and constrained numerical optimization, we determined the mathematically optimal weights for a ten-asset portfolio. This step formalizes the trade-off between expected return and portfolio variance.

**Analysis.**  
By visualizing the Efficient Frontier we observed how portfolio risk evolves as weights change. This geometric view makes clear that portfolio selection is fundamentally a problem of navigating a curved risk–return surface.

**Protection.**  
We then introduced the Black–Scholes framework to price a **protective put** on the optimized portfolio. The option acts as a form of insurance: the premium lowers the expected payoff slightly, but in return it imposes a floor that eliminates catastrophic downside risk. In effect, derivatives allow us to reshape the payoff distribution of the portfolio.

### Transition to Notebook 2_2

While we have applied these concepts to a portfolio of equities, the same mathematical principles appear in **fixed-income instruments**. Many bonds contain embedded options—such as call provisions—that alter their payoff structure.

In the next and final notebook we will examine how these embedded options affect **bond convexity and duration**, and demonstrate how Taylor approximations can be used to understand the curvature of bond price functions in response to changes in interest rates.